In [1]:
import findspark

In [2]:
findspark.init()

In [3]:
from pyspark.sql import SparkSession

In [9]:
spark = SparkSession.builder.appName("first code").master('local[3]').config('spark.driver.host','127.0.0.1')\
.config('spark.ui.port',0).getOrCreate()

In [10]:
spark

In [6]:
type(spark.sparkContext)

pyspark.context.SparkContext

In [11]:
ppl= spark.read.format('json').load(r'/home/AzureAdmin/data/people.json')   #eager execution    

In [12]:
type(ppl)

pyspark.sql.dataframe.DataFrame

In [13]:
ppl.printSchema()

root
 |-- age: long (nullable = true)
 |-- name: string (nullable = true)
 |-- pcode: string (nullable = true)



In [14]:
ppl.columns

['age', 'name', 'pcode']

In [15]:
ppl.schema

StructType([StructField('age', LongType(), True), StructField('name', StringType(), True), StructField('pcode', StringType(), True)])

In [16]:
ppl.show()

+----+-------+-----+
| age|   name|pcode|
+----+-------+-----+
|NULL|  Alice|94304|
|  30|Brayden|94304|
|  19|  Carla|10036|
|  46|  Diana| NULL|
|NULL|Etienne|94104|
+----+-------+-----+



In [17]:
acc_par = spark.read.format('parquet').load(r'/home/AzureAdmin/data/accounts_parquet/accounts_parquet')


In [18]:
acc_par.printSchema()

root
 |-- acct_num: integer (nullable = true)
 |-- acct_create_dt: long (nullable = true)
 |-- acct_close_dt: long (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- address: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- zipcode: string (nullable = true)
 |-- phone_number: string (nullable = true)
 |-- created: long (nullable = true)
 |-- modified: long (nullable = true)



In [19]:
acc_par.show()

+--------+--------------+-------------+----------+---------+--------------------+-------------+-----+-------+------------+-------------+-------------+
|acct_num|acct_create_dt|acct_close_dt|first_name|last_name|             address|         city|state|zipcode|phone_number|      created|     modified|
+--------+--------------+-------------+----------+---------+--------------------+-------------+-----+-------+------------+-------------+-------------+
|   32441| 1341603329000|         NULL|     Peter|  Zachary|  1891 Jessie Street|   Long Beach|   CA|  90743|  5628719002|1395174646000|1395174646000|
|   32442| 1345273687000|         NULL|     Duane|     Ruiz| 1023 Simpson Street|     Pasadena|   CA|  91051|  6262472463|1395174646000|1395174646000|
|   32443| 1347189332000|         NULL|   Desiree|    Beall|    4830 Davis Place|         Reno|   NV|  89468|  7758006922|1395174646000|1395174646000|
|   32444| 1353649861000|         NULL|     Molly|   Pinder|    708 Locust Court|    Las Vegas

In [20]:
acc_par.count()

97320

In [21]:
ppl.describe().show()

+-------+------------------+-------+------------------+
|summary|               age|   name|             pcode|
+-------+------------------+-------+------------------+
|  count|                 3|      5|                 4|
|   mean|31.666666666666668|   NULL|           73187.0|
| stddev|13.576941236277534|   NULL|42100.772233614276|
|    min|                19|  Alice|             10036|
|    max|                46|Etienne|             94304|
+-------+------------------+-------+------------------+



In [32]:
latlon = spark.read.format('csv').option('sep','\t')\
.option('inferSchema',True)\
.load(r'/home/AzureAdmin/data/latlon.tsv')

In [33]:
latlon.printSchema()

root
 |-- _c0: integer (nullable = true)
 |-- _c1: double (nullable = true)
 |-- _c2: double (nullable = true)



In [34]:
latlon.show(3)

+---+---------+----------+
|_c0|      _c1|       _c2|
+---+---------+----------+
|210|43.005895|-71.013202|
|211|43.005895|-71.013202|
|212|43.005895|-71.013202|
+---+---------+----------+
only showing top 3 rows



In [35]:
from pyspark.sql.types import *

In [36]:
sch = StructType([StructField("citycode",IntegerType(),True),\
                 StructField("lattitude",FloatType(),True),\
                 StructField("longitude",FloatType(),True)])

In [38]:
latlon = spark.read.format('csv').option('sep','\t')\      #lazy execution
.schema(sch).load(r'/home/AzureAdmin/data/latlon.tsv')

In [39]:
latlon.printSchema()


root
 |-- citycode: integer (nullable = true)
 |-- lattitude: float (nullable = true)
 |-- longitude: float (nullable = true)



In [40]:
latlon.show(4)

+--------+---------+---------+
|citycode|lattitude|longitude|
+--------+---------+---------+
|     210|43.005894| -71.0132|
|     211|43.005894| -71.0132|
|     212|43.005894| -71.0132|
|     213|43.005894| -71.0132|
+--------+---------+---------+
only showing top 4 rows



In [42]:
res = ppl.select ("name","age","pcode")
res.show()

+-------+----+-----+
|   name| age|pcode|
+-------+----+-----+
|  Alice|NULL|94304|
|Brayden|  30|94304|
|  Carla|  19|10036|
|  Diana|  46| NULL|
|Etienne|NULL|94104|
+-------+----+-----+



In [43]:
res = ppl.select(ppl["name"],ppl["age"],ppl["pcode"])
res.show()

+-------+----+-----+
|   name| age|pcode|
+-------+----+-----+
|  Alice|NULL|94304|
|Brayden|  30|94304|
|  Carla|  19|10036|
|  Diana|  46| NULL|
|Etienne|NULL|94104|
+-------+----+-----+



In [45]:
res = ppl.select (ppl.name,ppl.age,ppl.pcode)
res.show()

+-------+----+-----+
|   name| age|pcode|
+-------+----+-----+
|  Alice|NULL|94304|
|Brayden|  30|94304|
|  Carla|  19|10036|
|  Diana|  46| NULL|
|Etienne|NULL|94104|
+-------+----+-----+



In [46]:
from pyspark.sql.functions import col

In [47]:
res = ppl.select(col("name"),col("age"),col("pcode"))
res.show()

+-------+----+-----+
|   name| age|pcode|
+-------+----+-----+
|  Alice|NULL|94304|
|Brayden|  30|94304|
|  Carla|  19|10036|
|  Diana|  46| NULL|
|Etienne|NULL|94104|
+-------+----+-----+



In [48]:
res = ppl.where(col("age")>30)
res.show()

+---+-----+-----+
|age| name|pcode|
+---+-----+-----+
| 46|Diana| NULL|
+---+-----+-----+



In [50]:
acc_par.where(col("state") == 'CA').select(col("first_name"),col("state")).show()

+----------+-----+
|first_name|state|
+----------+-----+
|     Peter|   CA|
|     Duane|   CA|
|   Melissa|   CA|
|    Oliver|   CA|
|       Ben|   CA|
| Christina|   CA|
|      Jake|   CA|
|    Johnny|   CA|
|      Rita|   CA|
| Elizabeth|   CA|
|    Bobbie|   CA|
|   Cynthia|   CA|
|     Wanda|   CA|
|    Carole|   CA|
|    Amanda|   CA|
|   Phillip|   CA|
|     Erika|   CA|
|  Kimberly|   CA|
|   Claudia|   CA|
| Heriberto|   CA|
+----------+-----+
only showing top 20 rows



In [51]:
ppl.orderBy("age").show()

+----+-------+-----+
| age|   name|pcode|
+----+-------+-----+
|NULL|  Alice|94304|
|NULL|Etienne|94104|
|  19|  Carla|10036|
|  30|Brayden|94304|
|  46|  Diana| NULL|
+----+-------+-----+



In [52]:
ppl.orderBy(col("age").desc_nulls_last()).show()

+----+-------+-----+
| age|   name|pcode|
+----+-------+-----+
|  46|  Diana| NULL|
|  30|Brayden|94304|
|  19|  Carla|10036|
|NULL|  Alice|94304|
|NULL|Etienne|94104|
+----+-------+-----+



In [53]:
ppl.orderBy("pcode",col("age").desc_nulls_last()).show()

+----+-------+-----+
| age|   name|pcode|
+----+-------+-----+
|  46|  Diana| NULL|
|  19|  Carla|10036|
|NULL|Etienne|94104|
|  30|Brayden|94304|
|NULL|  Alice|94304|
+----+-------+-----+

